## Parte 1: Inspección de Espacios y Primer Contacto

In [13]:
import time
import gymnasium as gym
import miniwob
from miniwob.action import ActionTypes

# 1. Crear el entorno con renderizado visual para inspección
env = gym.make('miniwob/click-test-2-v1', render_mode='human')
obs, info = env.reset(seed=42)

print("--- OBSERVACIÓN ---")
print("Llaves en el diccionario de observación:", obs.keys())
print("Instrucción (utterance):", obs.get('utterance'))
print("Dimensiones de la captura (screenshot shape):", obs['screenshot'].shape)

print("\n--- ESPACIO DE ACCIONES ---")
print("Tipo de Action Space:", env.action_space)

env.close()

--- OBSERVACIÓN ---
Llaves en el diccionario de observación: dict_keys(['utterance', 'dom_elements', 'screenshot', 'fields'])
Instrucción (utterance): Click button ONE.
Dimensiones de la captura (screenshot shape): (210, 160, 3)

--- ESPACIO DE ACCIONES ---
Tipo de Action Space: Dict('action_type': Discrete(14), 'coords': Box(0.0, [160. 210.], (2,), float32), 'field': Discrete(20), 'key': Discrete(104), 'ref': Discrete(1000000), 'text': Unicode(1, 64))




- ¿Qué forma (shape) y tipo de datos (dtype) tiene la observación visual (screenshot)?
- ¿Qué aporta dom_elements frente a usar solo la imagen RGB?


In [14]:
shape = obs['screenshot'].shape
print("\n--- DIMENSIONES DE LA CAPTURA ---", shape)

tipo = obs['screenshot'].dtype
print("\n--- TIPO DE DATO DE LA CAPTURA ---", tipo)


--- DIMENSIONES DE LA CAPTURA --- (210, 160, 3)

--- TIPO DE DATO DE LA CAPTURA --- uint8


In [15]:
dom_elems = obs['dom_elements']
print("\n--- ELEMENTOS DEL DOM ---")
for i, elem in enumerate(dom_elems):
    print(f"Elemento {i}: {elem}")

from collections import Counter
print("\n--- RESUMEN ---")
print(f"Total de elementos DOM: {len(dom_elems)}")
print("Etiquetas:", dict(Counter(e['tag'] for e in dom_elems)))

print("\n--- BOTONES Y SU CENTRO DE CLIC ---")
for e in dom_elems:
    if e['tag'] == 'button':
        cx = float(e['left'][0] + e['width'][0] / 2)
        cy = float(e['top'][0] + e['height'][0] / 2)
        print(f"Botón {e['text']!r} (ref={e['ref']}, id={e['id']}, is_leaf={e['flags'][3]}): centro=({cx:.1f}, {cy:.1f})")


--- ELEMENTOS DEL DOM ---
Elemento 0: {'ref': 1, 'parent': 0, 'left': array([0.], dtype=float32), 'top': array([0.], dtype=float32), 'width': array([500.], dtype=float32), 'height': array([210.], dtype=float32), 'tag': 'body', 'text': '', 'value': '', 'id': '', 'classes': '', 'bg_color': array([0.33333334, 0.33333334, 0.33333334, 1.        ], dtype=float32), 'fg_color': array([0., 0., 0., 1.], dtype=float32), 'flags': array([1, 0, 0, 0], dtype=int8)}
Elemento 1: {'ref': 2, 'parent': 1, 'left': array([0.], dtype=float32), 'top': array([0.], dtype=float32), 'width': array([160.], dtype=float32), 'height': array([210.], dtype=float32), 'tag': 'div', 'text': '', 'value': '', 'id': 'wrap', 'classes': '', 'bg_color': array([1., 1., 1., 1.], dtype=float32), 'fg_color': array([0., 0., 0., 1.], dtype=float32), 'flags': array([0, 0, 0, 0], dtype=int8)}
Elemento 2: {'ref': 3, 'parent': 2, 'left': array([0.], dtype=float32), 'top': array([50.], dtype=float32), 'width': array([160.], dtype=float32

### Respuestas — Parte 1

**R1.** `shape = (210, 160, 3)` (alto × ancho × RGB) y `dtype = uint8` (0–255).

**R2.** Aporta semántica y estructura: `tag`/`text` identifican el elemento correcto, `ref` permite `CLICK_ELEMENT` y `left`/`top`/`width`/`height` dan el centro del clic. Con solo RGB el agente tendría que localizarlo visualmente.


## Parte 2: Agente Aleatorio vs. Entorno Web

10 episodios con `env.action_space.sample()` y `render_mode=None`. Las acciones inválidas para Selenium se ignoran; si no se acierta en 10 s, el episodio termina con recompensa −1.


In [16]:
import logging
import time

# miniwob registra un WARNING por cada CLICK_ELEMENT con ref inexistente
# (ruido esperado al muestrear acciones al azar); se eleva el nivel para
# mostrar solo errores reales.
logging.getLogger().setLevel(logging.ERROR)

env = gym.make('miniwob/click-test-2-v1', render_mode=None)
env.action_space.seed(42)  # muestreo reproducible

num_episodes = 10
rewards = []
durations = []
errores_totales = 0
MAX_INTENTOS = 500
LIMITE_SEGUNDOS = 30.0

for ep in range(num_episodes):
    obs, info = env.reset(seed=42 + ep)
    terminated = False
    truncated = False
    total_reward = 0.0
    steps = 0
    intentos = 0
    errores = 0
    t0 = time.time()

    while not (terminated or truncated):
        if intentos >= MAX_INTENTOS or (time.time() - t0) > LIMITE_SEGUNDOS:
            truncated = True  # corte de seguridad del laboratorio
            break
        intentos += 1
        # Muestreo de una acción aleatoria
        action = env.action_space.sample()
        try:
            obs, reward, terminated, truncated, info = env.step(action)
        except Exception:
            # Acción inválida para Selenium (p. ej. scroll fuera de pantalla): se ignora.
            errores += 1
            continue
        total_reward += reward
        steps += 1

    rewards.append(total_reward)
    durations.append(steps)
    errores_totales += errores
    print(f"Episodio {ep + 1:2d}: recompensa = {total_reward:+.3f} | pasos = {steps} | errores Selenium = {errores}")

print(f"\nRecompensa promedio: {sum(rewards)/num_episodes:.4f}")
print(f"Pasos promedio por episodio: {sum(durations)/num_episodes:.2f}")
exitos = sum(1 for r in rewards if r > 0)
print(f"Episodios exitosos (r > 0): {exitos}/{num_episodes} ({100*exitos/num_episodes:.0f}%)")
print(f"Errores Selenium ignorados en total: {errores_totales}")
print(f"Recompensas por episodio: {[round(r, 3) for r in rewards]}")
env.close()

Episodio  1: recompensa = -1.000 | pasos = 105 | errores Selenium = 6
Episodio  2: recompensa = -1.000 | pasos = 205 | errores Selenium = 10
Episodio  3: recompensa = -1.000 | pasos = 57 | errores Selenium = 6
Episodio  4: recompensa = -1.000 | pasos = 270 | errores Selenium = 10
Episodio  5: recompensa = -1.000 | pasos = 224 | errores Selenium = 18
Episodio  6: recompensa = +0.297 | pasos = 107 | errores Selenium = 9
Episodio  7: recompensa = +0.680 | pasos = 53 | errores Selenium = 4
Episodio  8: recompensa = +0.000 | pasos = 449 | errores Selenium = 51
Episodio  9: recompensa = -1.000 | pasos = 60 | errores Selenium = 7
Episodio 10: recompensa = -1.000 | pasos = 80 | errores Selenium = 7

Recompensa promedio: -0.6023
Pasos promedio por episodio: 161.00
Episodios exitosos (r > 0): 2/10 (20%)
Errores Selenium ignorados en total: 128
Recompensas por episodio: [-1.0, -1.0, -1.0, -1.0, -1.0, 0.297, 0.68, 0.0, -1.0, -1.0]


Preguntas:
- ¿Cuál fue la tasa de éxito o recompensa media obtenida por el agente aleatorio?
- En el Capítulo 14, el autor simplifica el espacio del ratón discretizándolo en una cuadrícula fija (grid de 10×10píxeles). ¿Por qué esta discretización facilita el entrenamiento en comparación con el espacio continuo original de coordenadas de pantalla?



### Respuestas — Parte 2

**R3.** Recompensa media **−0.6083** y **161.0 pasos/episodio**; solo **3/10** episodios con `r > 0`. Rendimiento equivalente al azar.

**R4.** Convierte el clic continuo `(x, y)` en **256 clases** (16×16 celdas de 10 px sobre 160×160): pasa de regresión a **clasificación** con softmax, reduce la varianza del gradiente y cada celda cubre el tamaño típico de un botón.


## Parte 3: Demostraciones Humanas

El RL puro falla con recompensa dispersa (`count-sides`, `tic-tac-toe`); las demostraciones añaden señal supervisada.

| Herramienta | Qué guarda |
| :--- | :--- |
| `miniwob.scripts.record` | Acciones + DOM, **sin píxeles** |
| `record_demo.py` (libro) | Acciones + DOM + **screenshot** |



In [18]:
import json
import pathlib
import subprocess
import sys
import urllib.request
from collections import Counter

import miniwob

print("--- Grabador oficial: python -m miniwob.scripts.record ---")
rec_path = pathlib.Path(miniwob.__file__).parent / "scripts" / "record.py"
print("Fuente instalada:", rec_path)
help_out = subprocess.run(
    [sys.executable, "-m", "miniwob.scripts.record", "--help"],
    capture_output=True, text=True, timeout=30,
)
if help_out.returncode == 0:
    print(help_out.stdout)
else:
    print("No se pudo ejecutar el grabador (falta `pip install bottle` en este entorno).")
    doc = rec_path.read_text().split('"""')[1].strip()
    print("Docstring del módulo:", doc)
    print("Uso según el código fuente: python -m miniwob.scripts.record <outdir> [-p PUERTO] [-g]")
    print("Lógica de guardado (Saver.save): recibe el JSON por POST /record y lo escribe")
    print("como <taskName>_<timestamp>.json; añade `?record=true` a la URL de la tarea.")

URL = ("https://raw.githubusercontent.com/PacktPublishing/"
       "Deep-Reinforcement-Learning-Hands-On-3E/main/Chapter14/"
       "demos/tic-tac-toe/tic-tac-toe_0423220647.json")
print("\n--- Demo real del libro (tic-tac-toe, ~28 KB) ---")
try:
    with urllib.request.urlopen(URL, timeout=20) as r:
        demo = json.loads(r.read())
except Exception as e:
    demo = None
    print(f"No se pudo descargar la demo ({e}).")

if demo is not None:
    print("Claves raíz:", list(demo.keys()))
    print("taskName:", demo["taskName"], "| utterance:", demo["utterance"])
    print("reward:", demo["reward"], "| rawReward:", demo["rawReward"])
    states = demo["states"]
    print(f"Número de estados: {len(states)}")
    tipos = Counter((s.get("action") or {}).get("type") for s in states)
    print("Tipos de evento:", dict(tipos))
    timings = Counter((s.get("action") or {}).get("timing") for s in states)
    print("timing (1 = antes de resolver, 3 = después):", dict(timings))
    s1 = states[1]
    print("Claves de un estado intermedio:", list(s1.keys()))
    print("Acción de ejemplo:", s1["action"])
    print("DOM raíz (árbol):", {k: (f"list[{len(v)}]" if k == "children" else v) for k, v in s1["dom"].items()})
    print("¿Todos los estados traen screenshot?", all("screenshot" in s for s in states))




--- Grabador oficial: python -m miniwob.scripts.record ---
Fuente instalada: /Users/marcosbautista/gym/lib/python3.11/site-packages/miniwob/scripts/record.py
usage: record.py [-h] [-p PORT] [-g] outdir

positional arguments:
  outdir                Directory to dump the demonstrations

options:
  -h, --help            show this help message and exit
  -p PORT, --port PORT  Open the connection at this port
  -g, --global-access   Allow global access to the server


--- Demo real del libro (tic-tac-toe, ~28 KB) ---
Claves raíz: ['taskName', 'utterance', 'states', 'reward', 'rawReward']
taskName: tic-tac-toe | utterance: Playing as 'X', win a game of tic-tac-toe.
reward: -1 | rawReward: -1
Número de estados: 23
Tipos de evento: {None: 1, 'mousedown': 8, 'mouseup': 6, 'click': 8}
timing (1 = antes de resolver, 3 = después): {None: 1, 1: 11, 3: 11}
Claves de un estado intermedio: ['time', 'action', 'dom', 'screenshot']
Acción de ejemplo: {'type': 'mousedown', 'x': 83, 'y': 125, 'timing': 1}

### PReguntas:
- ¿Qué componentes se guardan en el archivo de demostración (JSON) por cada paso ejecutado?
- ¿Cómo se incorpora la pérdida por imitación (aprendizaje supervisado) a la función de pérdida del método de Policy Gradients/Actor-Critic?




### Respuestas — Parte 3

**¿Qué guarda el JSON por paso? R5** La raíz tiene `taskName`, `utterance`, `reward` y `rawReward`; cada `states[]` guarda `time`, `action` (`type`, coordenadas/teclas, `timing`) y `dom` (árbol completo; el nodo usado lleva `recordingTarget`). El grabador oficial omite píxeles; `record_demo.py` añade `screenshot`.

**¿Cómo entra la imitación en la pérdida R6?** Con `L = L_AC + λ·L_imit`, con `L_imit = −log π_θ(a_demo | s_demo)`. En el libro se intercalan batches de demo (p = 0.5, primeros 25 000 pasos) que actualizan solo la política y comparten el tronco convolucional.


## Discusión POMDP — pérdida de la propiedad de Márkov

Márkov exige que s_t resuma todo el historial (El estado actual contiene resumida la informacion pasada), pero la observación (screenshot + DOM + utterance) no indica qué ocurrió antes. En `click-button-sequence` la pantalla puede ser idéntica tras el primer clic y la acción óptima cambia; en `tic-tac-toe` o `count-sides` el progreso solo está en la trayectoria. Por eso es un **POMDP** y una política reactiva `π(a|o)` no puede ser óptima: necesita un **estado de creencia** `b_t`.

**Soluciones:** apilar fotogramas/acciones, memoria recurrente (LSTM/GRU) o atención sobre el historial, flags de progreso y demostraciones humanas.


## Conclusiones

- Observación **multimodal** (RGB + DOM + utterance) pero **parcialmente observable**: el DOM da estructura, no historia.
- El agente aleatorio no resuelve la tarea: recompensa media **−0.61** y **2/10** éxitos.
- Claves del capítulo: **discretizar el ratón** (16×16) y usar **demostraciones humanas** contra la recompensa dispersa.
- Las tareas secuenciales requieren **memoria** (stacking, LSTM/atención o flags).
